# kaiming-uniform-sf-init — worked example 3: SF init for Conv2d — fan_in includes kernel spatial dimensions

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `kaiming-uniform-sf-init`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

For a Conv2d layer, `fan_in = in_channels * kernel_h * kernel_w` rather than just `in_channels`. Each output activation is the dot product over the full receptive patch, so the scale factor must account for the spatial extent of the kernel. A larger kernel shrinks `sf`, keeping the distribution appropriately narrow.

## Worked solution

Step 1: Compute `fan_in = in_channels * kernel_h * kernel_w`.

Step 2: Compute `sf = fan_in ** -0.5`.

Step 3: Sample `t.rand(out_channels, in_channels, kernel_h, kernel_w, generator=g)` and scale to `(-sf, +sf)`.

Step 4: Compare the `sf` for a 3×3 kernel vs a 1×1 kernel with the same channel count — the 3×3 `sf` should be 3× smaller, because `sqrt(9*in_ch) = 3*sqrt(in_ch)`.

In [ ]:
import torch as t

def kaiming_uniform_sf_conv2d(out_channels, in_channels, kernel_h, kernel_w, generator):
    fan_in = in_channels * kernel_h * kernel_w
    sf = fan_in ** -0.5
    raw = t.rand(out_channels, in_channels, kernel_h, kernel_w, generator=generator)
    return (raw * 2 - 1) * sf

g = t.Generator()

out_ch, in_ch = 32, 8

# 1x1 kernel
g.manual_seed(0)
w_1x1 = kaiming_uniform_sf_conv2d(out_ch, in_ch, 1, 1, g)
sf_1x1 = (in_ch * 1 * 1) ** -0.5

# 3x3 kernel
g.manual_seed(0)
w_3x3 = kaiming_uniform_sf_conv2d(out_ch, in_ch, 3, 3, g)
sf_3x3 = (in_ch * 3 * 3) ** -0.5

print(f'1x1 kernel: sf={sf_1x1:.4f}, max_abs={w_1x1.abs().max().item():.4f}')
print(f'3x3 kernel: sf={sf_3x3:.4f}, max_abs={w_3x3.abs().max().item():.4f}')
print(f'sf ratio (1x1 / 3x3): {sf_1x1 / sf_3x3:.2f}  (should be ~3.0)')

assert w_1x1.shape == (out_ch, in_ch, 1, 1)
assert w_3x3.shape == (out_ch, in_ch, 3, 3)
assert w_1x1.abs().max().item() <= sf_1x1 + 1e-6
assert w_3x3.abs().max().item() <= sf_3x3 + 1e-6
print('Shapes and bounds OK.')